# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

We construct the clean, leak-free feature vector from `data/raw/content_refresh_anonymized.csv`.
Following strict data engineering principles:
1. Filter rows to eligible content: `impressions_90d > 0` and `content_age_days >= 90`.
2. Deduplicate strictly on `content_id`.
3. Impute missing values with explicit domain logic (numerics $\to 0$, categoricals $\to \text{"unknown"}$).
4. Engineer non-linear log transformations for heavy-tailed traffic totals (`log1p`).
5. Engineer indicator flags (`has_clicks`, `has_ai_sessions`, `measurable_opportunity`) without peeking at the target.

In [1]:
import os
import pandas as pd
import numpy as np

data_path = "../../data/raw/content_refresh_anonymized.csv" if os.path.exists("../../data/raw/content_refresh_anonymized.csv") else "data/raw/content_refresh_anonymized.csv"
raw_df = pd.read_csv(data_path)

# Filter to mature content with search exposure
clean_df = raw_df[(raw_df["impressions_90d"] > 0) & (raw_df["content_age_days"] >= 90)].copy()
clean_df = clean_df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

# Define target
clean_df["is_declining_label"] = (clean_df["trend_direction"].str.lower() == "down").astype(int)

# Engineer non-leaky transformed features
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    clean_df[f"log_{col}"] = np.log1p(clean_df[col].clip(lower=0))

clean_df["has_clicks"] = (clean_df["clicks_90d"] > 0).astype(int)
clean_df["has_ai_sessions"] = (clean_df["ai_sessions_90d"] > 0).astype(int)
clean_df["measurable_opportunity"] = ((clean_df["impressions_90d"] >= 100) & (clean_df["sessions_90d"] > 0)).astype(int)

# Handle missing values
numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "impressions_prev_30d",
    "clicks_prev_30d", "sessions_prev_30d", "content_age_days", "age_tier_order",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct", "log_impressions_90d", "log_clicks_90d",
    "log_sessions_90d", "log_ai_sessions_90d", "has_clicks", "has_ai_sessions",
    "measurable_opportunity"
]

categorical_features = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "char_count_tier",
    "impression_tier", "position_tier"
]

clean_df[numeric_features] = clean_df[numeric_features].fillna(0)
clean_df[categorical_features] = clean_df[categorical_features].fillna("unknown")

print(f"Feature Vector Built: {clean_df.shape[0]:,} rows x {clean_df.shape[1]} columns")
print(f"Numeric feature count: {len(numeric_features)}")
print(f"Categorical feature count: {len(categorical_features)}")
print(f"Target distribution (is_declining_label): {clean_df['is_declining_label'].mean()*100:.1f}% positive")

Feature Vector Built: 30,000 rows x 52 columns
Numeric feature count: 33
Categorical feature count: 9
Target distribution (is_declining_label): 54.2% positive


## 2. Feature notes (meaning, missing, categorical, available-when?)

A rigorous inventory of feature definitions, missingness patterns, encoding treatments, and temporal availability.

In [2]:
# Feature metadata inventory
feature_notes = pd.DataFrame([
    {"Feature": "log_impressions_90d", "Type": "Numeric (log1p)", "Available": "Pre-decision (90d)", "Missing Policy": "Filled with 0", "Signal Rationale": "Core visibility scale"},
    {"Feature": "avg_position", "Type": "Numeric", "Available": "Pre-decision (90d)", "Missing Policy": "0 means no data", "Signal Rationale": "Page 1-2 vs deep ranking"},
    {"Feature": "ctr", "Type": "Numeric (%)", "Available": "Pre-decision (90d)", "Missing Policy": "Filled with 0", "Signal Rationale": "Click efficiency (scaled x100)"},
    {"Feature": "content_age_days", "Type": "Numeric", "Available": "At publish time", "Missing Policy": "None (complete)", "Signal Rationale": "Maturity / evergreen decay"},
    {"Feature": "days_since_last_update", "Type": "Numeric", "Available": "Pre-decision", "Missing Policy": "None (complete)", "Signal Rationale": "Staleness signal"},
    {"Feature": "impressions_prev_30d", "Type": "Numeric", "Available": "Pre-decision (days 31-60)", "Missing Policy": "Filled with 0", "Signal Rationale": "Baseline historical momentum"},
    {"Feature": "content_type", "Type": "Categorical", "Available": "At publish time", "Missing Policy": "Categorical string", "Signal Rationale": "Editorial format archetype"},
    {"Feature": "position_tier", "Type": "Categorical", "Available": "Pre-decision", "Missing Policy": "Discretized bucket", "Signal Rationale": "Non-linear ranking bracket"}
])

display(feature_notes)

,Feature,Type,Available,Missing Policy,Signal Rationale
0,log_impressions_90d,Numeric (log1p),Pre-decision (90d),Filled with 0,Core visibility scale
1,avg_position,Numeric,Pre-decision (90d),0 means no data,Page 1-2 vs deep ranking
2,ctr,Numeric (%),Pre-decision (90d),Filled with 0,Click efficiency (scaled x100)
3,content_age_days,Numeric,At publish time,None (complete),Maturity / evergreen decay
4,days_since_last_update,Numeric,Pre-decision,None (complete),Staleness signal
5,impressions_prev_30d,Numeric,Pre-decision (days 31-60),Filled with 0,Baseline historical momentum
6,content_type,Categorical,At publish time,Categorical string,Editorial format archetype
7,position_tier,Categorical,Pre-decision,Discretized bucket,Non-linear ranking bracket


## 3. The leakage hunt

We actively attack our model by conducting a **controlled leakage experiment**:
1. We train an honest Decision Tree using only legitimate pre-decision features.
2. We deliberately inject the suspected leaky column (`trend_pct`) into the training set.
3. We observe the pathological jump in Precision@50 and inspect the resulting decision rules to catch the leak red-handed.

In [3]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split

# Precision@50 evaluator
def p_at_50(scores, y):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(y)[order[:50]].mean()

y = clean_df["is_declining_label"].values

# 1. HONEST MODEL (Pre-decision features only)
honest_cols = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count"]
X_honest = clean_df[honest_cols].fillna(0)

tree_honest = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree_honest.fit(X_honest, y)
honest_p50 = p_at_50(tree_honest.predict_proba(X_honest)[:, 1], y)

# 2. LEAKY MODEL (Injecting trend_pct)
X_leaky = clean_df[honest_cols + ["trend_pct"]].fillna(0)
tree_leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree_leaky.fit(X_leaky, y)
leaky_p50 = p_at_50(tree_leaky.predict_proba(X_leaky)[:, 1], y)

print("LEAKAGE ATTACK AUDIT RESULTS:")
print(f"  Honest Tree Precision@50: {honest_p50:.3f}")
print(f"  Leaky Tree Precision@50:  {leaky_p50:.3f}  <-- ARTIFACT OF LEAKAGE!")

print("\n--- Leaky Tree Decision Rules (Reveals circular shortcut) ---")
print(export_text(tree_leaky, feature_names=honest_cols + ["trend_pct"]))
print("VERDICT: trend_pct isolates the threshold (-20%) that defines the label itself. Confirmed fatal leakage.")

LEAKAGE ATTACK AUDIT RESULTS:
  Honest Tree Precision@50: 0.740
  Leaky Tree Precision@50:  1.000  <-- ARTIFACT OF LEAKAGE!

--- Leaky Tree Decision Rules (Reveals circular shortcut) ---
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0

VERDICT: trend_pct isolates the threshold (-20%) that defines the label itself. Confirmed fatal leakage.


## 4. What I excluded and why

Every excluded field and its scientific disqualification justification:

In [4]:
exclusions = pd.DataFrame([
    {"Excluded Column": "trend_direction", "Reason for Exclusion": "Fatal Label Leakage: Exactly matches the target definition (is_declining_label = trend_direction == 'down')."},
    {"Excluded Column": "trend_pct", "Reason for Exclusion": "Fatal Mathematical Leakage: The exact continuous percentage calculation from which trend_direction is thresholded."},
    {"Excluded Column": "impressions_last_30d", "Reason for Exclusion": "Lookahead Leakage: Represents the evaluation outcome window used to calculate recent momentum."},
    {"Excluded Column": "clicks_last_30d", "Reason for Exclusion": "Lookahead Leakage: Downstream outcome occurring during the evaluation window."},
    {"Excluded Column": "sessions_last_30d", "Reason for Exclusion": "Lookahead Leakage: Downstream outcome occurring during the evaluation window."},
    {"Excluded Column": "content_id", "Reason for Exclusion": "Identifier Key: Pseudonymous entity key. Using as a feature causes arbitrary memorization."},
    {"Excluded Column": "client_id", "Reason for Exclusion": "Identifier Key: Reserved strictly for grouped train/test validation splits (client-holdout)."},
    {"Excluded Column": "provider_used", "Reason for Exclusion": "High Missingness / Non-generalizable: Metadata reflecting LLM provider at generation time."},
    {"Excluded Column": "model_used", "Reason for Exclusion": "High Missingness / Non-generalizable: Model version string without intrinsic SEO content signal."}
])

display(exclusions)

,Excluded Column,Reason for Exclusion
0,trend_direction,Fatal Label Leakage: Exactly matches the targe...
1,trend_pct,Fatal Mathematical Leakage: The exact continuo...
2,impressions_last_30d,Lookahead Leakage: Represents the evaluation o...
3,clicks_last_30d,Lookahead Leakage: Downstream outcome occurrin...
4,sessions_last_30d,Lookahead Leakage: Downstream outcome occurrin...
5,content_id,Identifier Key: Pseudonymous entity key. Using...
6,client_id,Identifier Key: Reserved strictly for grouped ...
7,provider_used,High Missingness / Non-generalizable: Metadata...
8,model_used,High Missingness / Non-generalizable: Model ve...


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.